### **GPT-2 pequeño: tokens, embeddings, decodificación y KV cache**

#### **Propósito**

Este cuaderno estudia un **modelo de lenguaje causal decoder-only** desde una perspectiva formal, experimental y orientada a sistemas. La meta no es solo ejecutar un pipeline con `transformers`, sino conectar cada bloque empírico con una formulación matemática precisa.

#### **Alcance y limitaciones del modelo elegido**

Trabajaremos con **GPT-2** o **DistilGPT-2** como modelos base por razones de costo y portabilidad. Esta elección permite estudiar con claridad la mecánica interna de un LLM causal, pero conviene fijar tres advertencias desde el inicio:

1. **No es un modelo instruct moderno**: no fue alineado mediante SFT, RLHF o DPO.
2. **No representa el estado del arte actual** en capacidad, seguridad ni seguimiento de instrucciones.
3. **Las salidas observadas deben interpretarse como evidencia sobre el mecanismo autoregresivo**, no como referencia de calidad final de los LLM contemporáneos.

El cuaderno recorre el siguiente flujo:

1. **Texto $\rightarrow$ tokens** mediante un tokenizador subword.
2. **Tokens $\rightarrow$ embeddings** en un espacio vectorial denso.
3. **Factorización autoregresiva** de la probabilidad conjunta.
4. **Predicción del siguiente token** a partir de logits.
5. **Decodificación** bajo distintas políticas de inferencia.
6. **KV cache** como mecanismo de eficiencia computacional.

#### **Notación**

Sea:
- $\Sigma^*$ el conjunto de cadenas sobre un alfabeto $\Sigma$,
- $\mathcal V$ el vocabulario discreto de tokens,
- $\tau: \Sigma^* \to \mathcal V^*$ el tokenizador,
- $x_{1:T} = (x_1,\dots,x_T)$ una secuencia de tokens,
- $\theta$ los parámetros del modelo.

En un modelo causal, la distribución conjunta se factoriza como

$$
p_\theta(x_{1:T})=\prod_{t=1}^{T} p_\theta(x_t \mid x_{<t}),
$$

donde $x_{<t}=(x_1,\dots,x_{t-1})$. Esta es la hipótesis probabilística central del cuaderno.


#### **0. Preparación del entorno**

Se usarán:
- `transformers`,
- `torch`,
- `matplotlib`,
- `pandas`.

##### **Comentario metodológico**

Este cuaderno está pensado para ejecutarse en GPU si está disponible, aunque puede correr en CPU con un modelo más pequeño como `distilgpt2`. El uso de un modelo preentrenado reduce el costo experimental y permite concentrarse en la semántica de los pasos del pipeline.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Si estás en Colab o un entorno limpio, descomenta:
# !pip install -q transformers datasets accelerate sentencepiece

import time
import math
from typing import List, Dict

import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2"   # opción ligera: "distilgpt2"

print(f"Dispositivo: {DEVICE}")
print(f"Modelo elegido: {MODEL_NAME}")


#### **1. Cargar tokenizer y modelo causal**

Usaremos un modelo **decoder-only**.

##### **Definición**

Un modelo causal implementa, en la práctica, aproximaciones de la forma

$$
p_\theta(x_{t+1}\mid x_{1:t}).
$$

En un Transformer decoder-only, esa dependencia se obtiene imponiendo una **restricción causal**: el token en la posición $t$ no puede atender a posiciones futuras mayores que $t$.

##### **Observación**

En esta sección cargamos:
- un **tokenizer** que realiza la aplicación discreta $\tau$,
- un **modelo causal** que define una familia de distribuciones condicionales sobre el vocabulario.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 no trae token de padding por defecto; reutilizamos eos si hace falta.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Tokenizer cargado.")
print("Vocab size:", tokenizer.vocab_size)
print("Embedding dim:", model.config.n_embd)
print("N capas:", model.config.n_layer)
print("Max positions:", model.config.n_positions)


#### **2. De texto a tokens**

Aquí inspeccionamos:
- texto original,
- tokens,
- identificadores (`ids`),
- longitud en tokens.

##### **Formalización**

El tokenizador define una aplicación

$$
\tau(s)= (x_1,\dots,x_T), \qquad x_t \in \mathcal V.
$$

La longitud relevante para el modelo no es el número de palabras, sino el número de **tokens**. Por eso, desde el punto de vista computacional, el costo de procesar una entrada depende de $T$ y no directamente del número de palabras ortográficas.

##### **Idea clave**

La desigualdad "tokens $\neq$ palabras" no es un detalle de implementación: modifica costo, granularidad de representación y comportamiento de la ventana de contexto.


In [ ]:
prompts = [
    "Los modelos de lenguaje generan texto token por token.",
    "Resume en dos líneas qué es un transformer causal.",
    "Explica brevemente la diferencia entre greedy decoding y top-p sampling."
]

rows = []
for text in prompts:
    encoded = tokenizer(text, add_special_tokens=False)
    ids = encoded["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    rows.append({
        "texto": text,
        "n_tokens": len(ids),
        "tokens": toks[:12],
        "ids": ids[:12]
    })

df_tokens = pd.DataFrame(rows)
df_tokens


In [ ]:
texto = prompts[0]
ids = tokenizer(texto, add_special_tokens=False)["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(ids)

print("Texto:")
print(texto)
print("\nTokens:")
print(tokens)
print("\nIDs:")
print(ids)

#### **Ejercicio**

Prueba tres frases distintas:
1. una corta,
2. una larga,
3. una con signos, números o fragmentos de código.

##### **Pregunta analítica**

¿Cómo cambia $T = |\tau(s)|$ según la estructura superficial del texto? Discute por qué dos secuencias con número similar de palabras pueden inducir longitudes en tokens bastante distintas.


In [ ]:
# Escribe aquí tus ejemplos
mis_frases = [
    "Ejemplo 1",
    "Ejemplo 2",
    "Ejemplo 3"
]

for frase in mis_frases:
    ids = tokenizer(frase, add_special_tokens=False)["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    print("Texto:", frase)
    print("N tokens:", len(ids))
    print("Tokens:", toks)


#### **3. Embeddings: de ids a vectores densos**

Una vez obtenida la secuencia discreta $x_{1:T}$, cada token se proyecta a un vector denso mediante una matriz de embeddings

$$
E \in \mathbb{R}^{|\mathcal V|\times d}.
$$

La representación embebida es

$$
X = [E[x_1],E[x_2],\dots,E[x_T]] \in \mathbb{R}^{T\times d}.
$$

##### **Lectura conceptual**

Los embeddings no son "significados completos" del token, sino parámetros entrenables que proveen una base de representación para el resto del modelo.

##### **Objetivo de esta sección**

- inspeccionar la matriz de embeddings,
- observar ejemplos de vectores,
- estudiar similitud coseno entre algunas filas.


In [ ]:
embedding_layer = model.get_input_embeddings()
W = embedding_layer.weight.detach().cpu()

print("Forma de la matriz de embeddings:", tuple(W.shape))
print("Primeras 3 filas, primeras 8 columnas:")
print(W[:3, :8])


##### **Similitud coseno entre embeddings**

Dado un par de vectores $u,v\in\mathbb{R}^d$, la similitud coseno se define como

$$
\cos(u,v)=\frac{u^\top v}{\|u\|\,\|v\|}.
$$

##### **Precaución interpretativa**

Una similitud alta en la **matriz de embeddings de entrada** no equivale automáticamente a una relación semántica profunda. Lo que observamos aquí es solo una parte del sistema de representación del modelo.

##### **Nota específica sobre GPT-2**

En GPT-2, los tokens BPE dependen también del patrón de espacios. Por eso es preferible inspeccionar **tokens reales obtenidos de una secuencia tokenizada**, en vez de inventar cadenas aisladas que podrían no corresponder a entradas frecuentes del vocabulario.


In [ ]:
sample_text = " language model token dog cat"
sample_ids = tokenizer(sample_text, add_special_tokens=False)["input_ids"]
sample_tokens = tokenizer.convert_ids_to_tokens(sample_ids)

# Nos quedamos con los primeros cinco tokens reales extraídos del tokenizer
tokens_demo = sample_tokens[:5]
ids_demo = sample_ids[:5]
vectors = W[ids_demo]

def cosine(u, v):
    return torch.dot(u, v) / (torch.norm(u) * torch.norm(v) + 1e-12)

sim = []
for i, tok_i in enumerate(tokens_demo):
    row = {
        "token_bpe": tok_i,
        "texto_decodificado": tokenizer.decode([ids_demo[i]])
    }
    for j, tok_j in enumerate(tokens_demo):
        row[f"sim_{j}"] = float(cosine(vectors[i], vectors[j]))
    sim.append(row)

pd.DataFrame(sim)


#### **Visual opcional: norma de embeddings**

La norma $\|E[x]\|_2$ de un embedding puede variar entre tokens. Esto recuerda que el espacio de embeddings no es uniforme y que distintas filas pueden tener magnitudes diferentes.

Formalmente, para un token $x$:

$$
\|E[x]\|_2 = \sqrt{\sum_{i=1}^{d} E[x]_i^2}.
$$


In [ ]:
norms = torch.norm(vectors, dim=1).numpy()
labels = [tokenizer.decode([idx]).strip() or tokenizer.decode([idx]) for idx in ids_demo]

plt.figure(figsize=(8, 4))
plt.bar(labels, norms)
plt.title("Norma de algunos embeddings de entrada")
plt.xlabel("Token decodificado")
plt.ylabel("Norma L2")
plt.show()


#### **4. Predicción del siguiente token**

Dado un prompt tokenizado $x_{1:T}$, el modelo produce **logits** sobre el vocabulario para la última posición.

##### **Definición**

Sea $z_T \in \mathbb{R}^{|\mathcal V|}$ el vector de logits del último paso. La distribución sobre el siguiente token se obtiene mediante softmax:

$$
p_\theta(v\mid x_{1:T})=
\frac{\exp(z_{T,v})}{\sum_{u\in\mathcal V}\exp(z_{T,u})}.
$$

##### **Lo que veremos**

Inspeccionaremos el **top-k** de candidatos para distinguir entre:
- la distribución local de un solo paso,
- y la dinámica completa de una generación de varios pasos.


In [ ]:
def top_next_tokens(prompt: str, k: int = 10) -> pd.DataFrame:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        values, indices = torch.topk(probs, k=k, dim=-1)

    tokens = tokenizer.convert_ids_to_tokens(indices[0].tolist())
    texts = [tokenizer.decode([idx]) for idx in indices[0].tolist()]

    return pd.DataFrame({
        "token": tokens,
        "texto_decodificado": texts,
        "probabilidad": values[0].detach().cpu().numpy()
    })

prompt = "La inteligencia artificial puede"
top_next_tokens(prompt, k=10)

#### **Preguntas para discutir**

- ¿Por qué algunos tokens decodificados parecen fragmentos o subcadenas?
- ¿Qué diferencia conceptual hay entre inspeccionar $p_\theta(\cdot\mid x_{1:T})$ en un solo paso y generar una secuencia completa $\hat{x}_{T+1:T+m}$?
- ¿Cómo puede acumularse el error local a través de múltiples iteraciones autoregresivas?.


#### **5. Generación autoregresiva y estrategias de decodificación**

En inferencia, el modelo itera el mapa

$$
x_{1:t}\mapsto p_\theta(x_{t+1}\mid x_{1:t}),
$$

y luego selecciona un token $\hat{x}_{t+1}$ según una política de decodificación.

#### **Estrategias consideradas**

**Greedy**
$$
\hat{x}_{t+1}=\arg\max_{v\in\mathcal V} p_\theta(v\mid x_{1:t}).
$$

**Temperatura**
$$
p^{(\tau)}(v\mid x_{1:t})=
\frac{\exp(z_v/\tau)}{\sum_{u\in\mathcal V}\exp(z_u/\tau)}.
$$

**Top-k**  
Se restringe el muestreo a los $k$ tokens con mayor probabilidad.

**Top-p**  
Se restringe el muestreo al conjunto mínimo $S_p$ tal que

$$
\sum_{v\in S_p} p_\theta(v\mid x_{1:t}) \ge p.
$$



In [ ]:
def generate_text(
    prompt: str,
    max_new_tokens: int = 40,
    do_sample: bool = False,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
    use_cache: bool = True
) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k if top_k > 0 else None,
            top_p=top_p,
            use_cache=use_cache,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

prompt = "Explica en dos líneas qué es un modelo de lenguaje causal:"
print("PROMPT:\n", prompt)


In [ ]:
configs = [
    {"nombre": "Greedy", "do_sample": False, "temperature": 1.0, "top_k": 0, "top_p": 1.0},
    {"nombre": "Sampling T=0.8", "do_sample": True, "temperature": 0.8, "top_k": 0, "top_p": 1.0},
    {"nombre": "Top-k=40", "do_sample": True, "temperature": 1.0, "top_k": 40, "top_p": 1.0},
    {"nombre": "Top-p=0.9", "do_sample": True, "temperature": 1.0, "top_k": 0, "top_p": 0.9},
]

results = []
for cfg in configs:
    text = generate_text(
        prompt,
        max_new_tokens=50,
        do_sample=cfg["do_sample"],
        temperature=cfg["temperature"],
        top_k=cfg["top_k"],
        top_p=cfg["top_p"],
        use_cache=True
    )
    results.append({"estrategia": cfg["nombre"], "salida": text})

pd.DataFrame(results)


#### **Actividad**

Cambia el prompt por una tarea más abierta o más técnica y compara las salidas.

##### **Preguntas guía**

- ¿Qué política produce la salida más estable?
- ¿Cuál produce mayor diversidad léxica?
- ¿En qué tipo de aplicación convendría maximizar control en lugar de diversidad?.


#### **6. Ventana de contexto y presupuesto de tokens**

La ventana de contexto puede modelarse como una restricción de longitud:

$$
T_{\text{prompt}} + T_{\text{historial}} + T_{\text{documentos}} + T_{\text{respuesta}}
\le L_{\max}.
$$

##### **Lectura práctica**

El presupuesto total de tokens se reparte entre:
- instrucción,
- historial conversacional,
- contexto externo recuperado,
- y la continuación generada.


Diseñar bien el contexto no es solo una cuestión estilística, es una decisión de asignación de un recurso limitado.

##### **Nota de complejidad**

En atención densa, el costo de procesar una secuencia de longitud $T$ crece aproximadamente de forma cuadrática en $T$:

$$
\text{costo de atención} \propto T^2.
$$

Por eso, aumentar contexto puede mejorar cobertura de evidencia, pero también incrementa memoria, latencia y costo computacional.


In [ ]:
def count_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

contextos = {
    "prompt_corto": "Resume el texto.",
    "prompt_medio": "Resume el texto en tres oraciones y destaca la idea principal.",
    "prompt_largo": (
        "Eres un asistente técnico. Resume el texto en tres oraciones, "
        "identifica el concepto central y menciona una limitación práctica."
    ),
}

for nombre, txt in contextos.items():
    print(f"{nombre}: {count_tokens(txt)} tokens")


#### **Historial y contexto compuesto**

Cuando concatenamos varios fragmentos, el costo total es la suma de sus longitudes tokenizadas:

$$
|\tau(s_1 \oplus s_2 \oplus \cdots \oplus s_n)|
$$

que, en la práctica, debe mantenerse por debajo de la ventana máxima del modelo.


In [ ]:
historial = [
    "Usuario: ¿Qué es un LLM?",
    "Asistente: Un modelo que predice tokens de manera autoregresiva.",
    "Usuario: ¿Y qué significa autoregresivo?"
]

prompt_final = "\n".join(historial) + "\nAsistente:"
n = count_tokens(prompt_final)

print(prompt_final)
print("\nTokens totales del historial + prompt:", n)
print("Máximo de posiciones del modelo:", model.config.n_positions)


#### **7. KV cache: inferencia eficiente**

En un Transformer autoregresivo, volver a computar toda la atención en cada paso es costoso. El **KV cache** reutiliza las proyecciones de claves y valores ya calculadas para los tokens previos.

##### **Idea formal**

Si en cada capa ya disponemos de:
- claves $K_{1:t}$,
- valores $V_{1:t}$,

al predecir el token $t+1$ no hace falta recomputar desde cero las representaciones previas.

##### **Consecuencia computacional**

El cache reduce trabajo redundante y mejora latencia, sobre todo cuando la secuencia generada crece.

##### **Hipótesis experimental**

Compararemos tiempos con:
- `use_cache=True`,
- `use_cache=False`.

En GPU, la ejecución puede ser asíncrona. Por eso sincronizamos explícitamente el dispositivo antes y después de medir tiempos para que la comparación sea más fiel.


In [ ]:
def _sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def timed_generation(prompt: str, use_cache: bool, reps: int = 3, max_new_tokens: int = 80) -> Dict[str, float]:
    # fase de calentamiento
    _ = generate_text(prompt, max_new_tokens=10, do_sample=False, use_cache=use_cache)
    _sync_device()

    times = []
    for _ in range(reps):
        _sync_device()
        start = time.perf_counter()
        _ = generate_text(prompt, max_new_tokens=max_new_tokens, do_sample=False, use_cache=use_cache)
        _sync_device()
        end = time.perf_counter()
        times.append(end - start)

    return {
        "use_cache": use_cache,
        "promedio_segundos": sum(times) / len(times),
        "min_segundos": min(times),
        "max_segundos": max(times)
    }

prompt_cache = "Explica brevemente por qué el KV cache acelera la generación autoregresiva."
res_true = timed_generation(prompt_cache, use_cache=True)
res_false = timed_generation(prompt_cache, use_cache=False)

pd.DataFrame([res_true, res_false])


#### **Visualización de tiempos**

La comparación no constituye una prueba teórica completa, pero sí ofrece un indicador empírico de la ganancia de eficiencia que aporta el cache en generación autoregresiva.


In [ ]:
cache_df = pd.DataFrame([res_true, res_false])

plt.figure(figsize=(6, 4))
plt.bar(cache_df["use_cache"].astype(str), cache_df["promedio_segundos"])
plt.title("Tiempo promedio de generación")
plt.xlabel("use_cache")
plt.ylabel("segundos")
plt.show()


#### **Preguntas para discusión**

- ¿Qué se reutiliza exactamente en el KV cache?
- ¿Por qué el beneficio crece con la longitud de la secuencia generada?
- ¿Cómo se conecta esto con latencia, throughput y costo de serving?.


#### **8. Prompt libre vs prompt instruccional**

Aunque GPT-2 no es un modelo instruct moderno, puede observarse cómo cambia la distribución inducida por el prompt cuando se modifica su estructura discursiva.

##### **Idea**

El modelo no "entiende instrucciones" en el sentido de un sistema alineado vía SFT o RLHF, pero sí responde a regularidades de formato presentes en sus datos de preentrenamiento.


Si el comportamiento parece inestable o poco obediente, eso no invalida el experimento: más bien refleja la diferencia entre un **modelo base** y un **modelo instruccionalmente ajustado**.

##### **Pregunta**

¿Qué diferencia se observa entre un prefijo abierto y un prefijo con estructura explícita de instrucción-respuesta?.


In [ ]:
prompt_libre = "Transformers y modelos de lenguaje"
prompt_instruccion = (
    "Instrucción: explica en lenguaje sencillo qué es un transformer causal.\n"
    "Respuesta:"
)

salida_libre = generate_text(prompt_libre, max_new_tokens=60, do_sample=True, temperature=0.8, top_p=0.95)
salida_inst = generate_text(prompt_instruccion, max_new_tokens=60, do_sample=True, temperature=0.8, top_p=0.95)

print("Prompt libre")
print(salida_libre)
print("\nPrompt con formato de instrucción")
print(salida_inst)


#### **9. Una misma base, varias tareas**

Un mismo modelo causal puede reutilizarse con diferentes formatos de contexto:
- continuación,
- resumen,
- clasificación verbal,
- explicación.

##### **Interpretación**

Muchas tareas pueden verse como instancias de **modelado condicional de continuaciones**. No cambia la arquitectura subyacente; cambia el modo en que construimos el contexto $x_{1:t}$.


In [ ]:
tareas = {
    "continuacion": "La tokenización por subpalabras es útil porque",
    "resumen": "Resume en una oración qué es la inferencia autoregresiva:\n",
    "clasificacion_verbal": (
        "Clasifica el sentimiento del siguiente texto como POSITIVO o NEGATIVO.\n"
        "Texto: 'El laboratorio estuvo claro y bien organizado.'\n"
        "Etiqueta:"
    ),
    "explicacion": "Explica qué diferencia hay entre greedy decoding y sampling:\n"
}

for nombre, p in tareas.items():
    print("TAREA:", nombre.upper())
    print(generate_text(p, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.95))
    print()


In [ ]:
# Tus respuestas
# Añade aquí nuevas pruebas, prompts, tablas o gráficos.

